# Data Preparation — Dirty Cafe Sales

## 1. Memuat Dataset dan Menyiapkan Salinan Data

Tahap Data Preparation dilakukan untuk memperbaiki kualitas data agar dataset siap digunakan dalam proses analisis. Proses ini mencakup penanganan nilai tidak valid, penyesuaian tipe data, pemeriksaan konsistensi transaksi, dan penanganan nilai kosong.

Dataset asli dimuat terlebih dahulu, kemudian disalin ke variabel `df_clean` agar perubahan selama proses pembersihan tidak memengaruhi data asli.

In [1]:
import pandas as pd
import numpy as np

# Memuat dataset asli
df = pd.read_csv("../data/dirty_cafe_sales.csv")

# Membuat salinan untuk proses pembersihan
df_clean = df.copy()

# Memeriksa ukuran dataset
print("Ukuran dataset:", df_clean.shape)

df_clean.head()

Ukuran dataset: (10000, 8)


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


## 2. Mengidentifikasi Nilai Tidak Valid

Dataset diperiksa untuk menemukan penanda `UNKNOWN` dan `ERROR`. Kedua nilai tersebut menunjukkan bahwa informasi pada suatu kolom tidak tersedia atau tidak valid untuk digunakan secara langsung dalam analisis. Nilai tersebut akan diubah menjadi nilai kosong agar dapat ditangani secara konsisten pada tahap berikutnya.

In [2]:
# Menghitung UNKNOWN, ERROR, dan nilai kosong sebelum penanganan
for kolom in df_clean.columns:
    nilai = df_clean[kolom].astype("string").str.strip().str.upper()

    print(
        f"{kolom}: "
        f"UNKNOWN={nilai.eq('UNKNOWN').sum()}, "
        f"ERROR={nilai.eq('ERROR').sum()}, "
        f"Kosong={df_clean[kolom].isna().sum()}"
    )

Transaction ID: UNKNOWN=0, ERROR=0, Kosong=0
Item: UNKNOWN=344, ERROR=292, Kosong=333
Quantity: UNKNOWN=171, ERROR=170, Kosong=138
Price Per Unit: UNKNOWN=164, ERROR=190, Kosong=179
Total Spent: UNKNOWN=165, ERROR=164, Kosong=173
Payment Method: UNKNOWN=293, ERROR=306, Kosong=2579
Location: UNKNOWN=338, ERROR=358, Kosong=3265
Transaction Date: UNKNOWN=159, ERROR=142, Kosong=159


## 3. Menangani Nilai UNKNOWN dan ERROR

Nilai `UNKNOWN` dan `ERROR` pada dataset diganti menjadi nilai kosong (`NaN`). Langkah ini dilakukan agar nilai yang tidak valid dapat dikenali secara konsisten dan ditangani pada proses berikutnya.

Proses dilakukan pada salinan dataset `df_clean` tanpa mengubah dataset asli.

In [3]:
df_clean = df_clean.replace(
    to_replace=r"^\s*(UNKNOWN|ERROR)\s*$",
    value=np.nan,
    regex=True
)

print("Jumlah nilai kosong setelah penggantian:")
print(df_clean.isna().sum())

Jumlah nilai kosong setelah penggantian:
Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64


## 4. Memeriksa Hasil Penggantian

Pemeriksaan dilakukan untuk memastikan nilai `UNKNOWN` dan `ERROR` telah dikonversi menjadi nilai kosong. Jumlah nilai kosong pada setiap kolom dibandingkan dengan kondisi sebelum proses penggantian.

In [4]:
print("Jumlah baris:", df_clean.shape[0])
print("Jumlah kolom:", df_clean.shape[1])

print("\nContoh data setelah penggantian:")
display(df_clean.head(10))

Jumlah baris: 10000
Jumlah kolom: 8

Contoh data setelah penggantian:


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,NaN,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,NaN,NaN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
5,TXN_2602893,Smoothie,5,4.0,20.0,Credit Card,NaN,2023-03-31
6,TXN_4433211,NaN,3,3.0,9.0,NaN,Takeaway,2023-10-06
7,TXN_6699534,Sandwich,4,4.0,16.0,Cash,NaN,2023-10-28
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5,4.0,20.0,NaN,In-store,2023-12-31


## 5. Mengubah Tipe Data

Kolom `Quantity`, `Price Per Unit`, dan `Total Spent` perlu diubah menjadi tipe numerik agar dapat digunakan dalam perhitungan. Sementara itu, kolom `Transaction Date` diubah menjadi format tanggal agar dapat digunakan untuk analisis tren penjualan berdasarkan waktu.

Nilai yang tidak dapat dikonversi akan dijadikan nilai kosong untuk diperiksa lebih lanjut.

In [5]:
# Mengubah kolom menjadi tipe numerik
kolom_numerik = [
    "Quantity",
    "Price Per Unit",
    "Total Spent"
]

for kolom in kolom_numerik:
    df_clean[kolom] = pd.to_numeric(
        df_clean[kolom],
        errors="coerce"
    )

# Mengubah kolom menjadi format tanggal
df_clean["Transaction Date"] = pd.to_datetime(
    df_clean["Transaction Date"],
    errors="coerce"
)

# Memeriksa tipe data setelah konversi
df_clean.dtypes

Transaction ID                 str
Item                           str
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method                 str
Location                       str
Transaction Date    datetime64[us]
dtype: object

## 6. Memeriksa Nilai Kosong Setelah Konversi

Setelah tipe data diubah, jumlah nilai kosong diperiksa kembali. Hal ini dilakukan karena nilai yang tidak sesuai dengan format angka atau tanggal dapat berubah menjadi nilai kosong selama proses konversi.

In [6]:
print("Jumlah nilai kosong setelah konversi:")
print(df_clean.isna().sum())

Jumlah nilai kosong setelah konversi:
Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64


## 7. Memeriksa Konsistensi Total Transaksi

Pemeriksaan ini bertujuan untuk mengetahui apakah nilai `Total Spent` sesuai dengan hasil perkalian `Quantity` dan `Price Per Unit`. Langkah ini membantu mengidentifikasi transaksi yang nilainya tidak konsisten.

Pemeriksaan hanya dilakukan pada baris yang memiliki ketiga nilai tersebut agar hasilnya tidak dipengaruhi oleh nilai kosong.

In [7]:
# Memilih baris dengan ketiga nilai tersedia
data_lengkap = df_clean.dropna(
    subset=["Quantity", "Price Per Unit", "Total Spent"]
).copy()

# Menghitung total berdasarkan jumlah dan harga
data_lengkap["Calculated Total"] = (
    data_lengkap["Quantity"] * data_lengkap["Price Per Unit"]
)

# Membandingkan total tercatat dengan hasil perhitungan
data_lengkap["Total Match"] = (
    data_lengkap["Total Spent"].round(2)
    == data_lengkap["Calculated Total"].round(2)
)

print("Jumlah transaksi yang diperiksa:", len(data_lengkap))
print("Total sesuai:", data_lengkap["Total Match"].sum())
print("Total tidak sesuai:", (~data_lengkap["Total Match"]).sum())

Jumlah transaksi yang diperiksa: 8544
Total sesuai: 8544
Total tidak sesuai: 0


## 8. Melihat Contoh Transaksi yang Tidak Konsisten

Beberapa transaksi yang memiliki perbedaan antara `Total Spent` dan hasil perhitungan ditampilkan untuk membantu mengidentifikasi pola ketidaksesuaian data sebelum menentukan tindakan perbaikan.

In [13]:
data_lengkap.loc[
    data_lengkap["Total Match"],
    [
        "Transaction ID",
        "Quantity",
        "Price Per Unit",
        "Total Spent",
        "Calculated Total"
    ]
].head(10)

,Transaction ID,Quantity,Price Per Unit,Total Spent,Calculated Total
0,TXN_1961373,2.0,2.0,4.0,4.0
1,TXN_4977031,4.0,3.0,12.0,12.0
3,TXN_7034554,2.0,5.0,10.0,10.0
4,TXN_3160411,2.0,2.0,4.0,4.0
5,TXN_2602893,5.0,4.0,20.0,20.0
6,TXN_4433211,3.0,3.0,9.0,9.0
7,TXN_6699534,4.0,4.0,16.0,16.0
8,TXN_4717867,5.0,3.0,15.0,15.0
9,TXN_2064365,5.0,4.0,20.0,20.0
10,TXN_2548360,5.0,5.0,25.0,25.0


## 9. Menangani Nilai Kosong

Setelah nilai `UNKNOWN` dan `ERROR` diubah menjadi nilai kosong (`NaN`), langkah berikutnya adalah menentukan penanganan nilai kosong pada setiap kolom. Penanganan dilakukan dengan mempertimbangkan fungsi kolom dan informasi yang masih tersedia.

Nilai kosong pada kolom kategori akan diperiksa berdasarkan frekuensi kemunculan nilai, sedangkan kolom numerik dan tanggal akan dianalisis sebelum menentukan metode pengisian atau penghapusan.

In [14]:
# Menghitung jumlah nilai kosong
jumlah_kosong = df_clean.isna().sum()

# Menghitung persentase nilai kosong
persen_kosong = (
    df_clean.isna().mean() * 100
).round(2)

# Menampilkan ringkasan nilai kosong
ringkasan_kosong = pd.DataFrame({
    "Jumlah Kosong": jumlah_kosong,
    "Persentase Kosong (%)": persen_kosong
})

display(ringkasan_kosong)

,Jumlah Kosong,Persentase Kosong (%)
Transaction ID,0,0.00
Item,969,9.69
Quantity,479,4.79
Price Per Unit,533,5.33
Total Spent,502,5.02
Payment Method,3178,31.78
Location,3961,39.61
Transaction Date,460,4.60


## 10. Memeriksa Kemungkinan Pemulihan Nilai Kosong

Sebelum mengisi atau menghapus nilai kosong, dilakukan pemeriksaan untuk mengetahui apakah informasi pada kolom lain dapat membantu memulihkan data. Pemeriksaan difokuskan pada kolom `Item`, `Quantity`, `Price Per Unit`, dan `Total Spent`.

Nilai kosong tidak langsung diisi dengan rata-rata atau modus karena pengisian yang tidak tepat dapat mengubah makna transaksi.

In [15]:
# Menampilkan 10 transaksi yang memiliki nilai kosong
baris_kosong = df_clean[
    df_clean.isna().any(axis=1)
]

display(baris_kosong.head(10))

print("Jumlah baris dengan minimal satu nilai kosong:", len(baris_kosong))

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
2,TXN_4271903,Cookie,4.0,1.0,NaN,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,NaN,NaN,2023-04-27
5,TXN_2602893,Smoothie,5.0,4.0,20.0,Credit Card,NaN,2023-03-31
6,TXN_4433211,NaN,3.0,3.0,9.0,NaN,Takeaway,2023-10-06
7,TXN_6699534,Sandwich,4.0,4.0,16.0,Cash,NaN,2023-10-28
8,TXN_4717867,NaN,5.0,3.0,15.0,NaN,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5.0,4.0,20.0,NaN,In-store,2023-12-31
11,TXN_3051279,Sandwich,2.0,4.0,8.0,Credit Card,Takeaway,NaT
13,TXN_9437049,Cookie,5.0,1.0,5.0,NaN,Takeaway,2023-06-01
14,TXN_8915701,NaN,2.0,1.5,3.0,NaN,In-store,2023-03-21


Jumlah baris dengan minimal satu nilai kosong: 6911


## 11. Memulihkan Total Transaksi

Nilai `Total Spent` yang kosong dapat dihitung jika `Quantity` dan `Price Per Unit` tersedia. Perhitungan hanya dilakukan pada baris yang memenuhi kedua kondisi tersebut agar tidak menghasilkan nilai berdasarkan perkiraan.

In [16]:
# Menghitung jumlah Total Spent yang kosong
# tetapi Quantity dan Price Per Unit tersedia
kondisi_total = (
    df_clean["Total Spent"].isna()
    & df_clean["Quantity"].notna()
    & df_clean["Price Per Unit"].notna()
)

print(
    "Total Spent kosong yang bisa dihitung:",
    kondisi_total.sum()
)

# Mengisi Total Spent berdasarkan Quantity x Price Per Unit
df_clean.loc[kondisi_total, "Total Spent"] = (
    df_clean.loc[kondisi_total, "Quantity"]
    * df_clean.loc[kondisi_total, "Price Per Unit"]
)

print(
    "Total Spent kosong setelah pemulihan:",
    df_clean["Total Spent"].isna().sum()
)

Total Spent kosong yang bisa dihitung: 462
Total Spent kosong setelah pemulihan: 40


## 12. Memeriksa Konsistensi Harga Produk

Pemeriksaan ini bertujuan untuk mengetahui harga yang tercatat pada setiap jenis produk. Informasi tersebut dapat digunakan untuk menilai apakah nilai `Price Per Unit` yang kosong bisa dipulihkan berdasarkan produk yang sama.

In [17]:
# Melihat jumlah harga unik dan variasi harga setiap produk
ringkasan_harga = (
    df_clean.dropna(subset=["Item", "Price Per Unit"])
    .groupby("Item")["Price Per Unit"]
    .agg(
        Jumlah_Data="count",
        Jumlah_Harga_Unik="nunique",
        Harga_Minimum="min",
        Harga_Maksimum="max",
        Harga_Paling_Sering=lambda x: x.mode().iloc[0]
    )
    .reset_index()
)

display(ringkasan_harga)

,Item,Jumlah_Data,Jumlah_Harga_Unik,Harga_Minimum,Harga_Maksimum,Harga_Paling_Sering
0,Cake,1085,1,3.0,3.0,3.0
1,Coffee,1108,1,2.0,2.0,2.0
2,Cookie,1026,1,1.0,1.0,1.0
3,Juice,1110,1,3.0,3.0,3.0
4,Salad,1082,1,5.0,5.0,5.0
5,Sandwich,1082,1,4.0,4.0,4.0
6,Smoothie,1036,1,4.0,4.0,4.0
7,Tea,1023,1,1.5,1.5,1.5


## 13. Mengisi Harga Berdasarkan Jenis Produk

Berdasarkan hasil pemeriksaan, setiap produk memiliki satu harga unik pada data yang tersedia. Oleh karena itu, nilai `Price Per Unit` yang kosong akan diisi menggunakan harga produk yang sama.

Jika nama produk juga kosong atau tidak memiliki informasi harga yang dapat digunakan, nilai tersebut akan tetap kosong.

In [18]:
# Membuat pemetaan produk ke harga
pemetaan_harga = (
    df_clean.dropna(subset=["Item", "Price Per Unit"])
    .drop_duplicates(subset=["Item"])
    .set_index("Item")["Price Per Unit"]
)

# Menentukan baris dengan harga kosong dan nama produk tersedia
kondisi_harga = (
    df_clean["Price Per Unit"].isna()
    & df_clean["Item"].notna()
)

print("Harga kosong yang bisa dipulihkan:", kondisi_harga.sum())

# Mengisi harga berdasarkan nama produk
df_clean.loc[kondisi_harga, "Price Per Unit"] = (
    df_clean.loc[kondisi_harga, "Item"].map(pemetaan_harga)
)

print(
    "Harga kosong setelah pemulihan:",
    df_clean["Price Per Unit"].isna().sum()
)

Harga kosong yang bisa dipulihkan: 479
Harga kosong setelah pemulihan: 54


## 14. Memeriksa Nilai Quantity yang Kosong

Kolom `Quantity` menunjukkan jumlah produk dalam suatu transaksi. Nilai kosong pada kolom ini perlu diperiksa bersama kolom `Item`, `Price Per Unit`, dan `Total Spent` untuk mengetahui apakah jumlah barang dapat dipulihkan secara logis.

Nilai `Quantity` tidak langsung diisi dengan rata-rata atau modus karena hal tersebut dapat mengubah jumlah transaksi yang sebenarnya.

In [19]:
# Memeriksa baris dengan Quantity kosong
quantity_kosong = df_clean[
    df_clean["Quantity"].isna()
]

display(
    quantity_kosong[
        [
            "Transaction ID",
            "Item",
            "Quantity",
            "Price Per Unit",
            "Total Spent"
        ]
    ].head(15)
)

print("Jumlah Quantity kosong:", df_clean["Quantity"].isna().sum())

# Menghitung berapa baris yang memiliki Item, harga,
# dan total transaksi meskipun Quantity kosong
bisa_diperiksa = quantity_kosong[
    quantity_kosong["Item"].notna()
    & quantity_kosong["Price Per Unit"].notna()
    & quantity_kosong["Total Spent"].notna()
]

print(
    "Quantity kosong dengan informasi pendukung lengkap:",
    len(bisa_diperiksa)
)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent
20,TXN_3522028,Smoothie,NaN,4.0,20.0
55,TXN_5522862,Cookie,NaN,1.0,2.0
57,TXN_2080895,Cake,NaN,3.0,3.0
66,TXN_8501819,Juice,NaN,3.0,6.0
117,TXN_2148617,Juice,NaN,3.0,9.0
153,TXN_6541415,NaN,NaN,3.0,12.0
177,TXN_1896955,Salad,NaN,5.0,25.0
178,TXN_5075974,Smoothie,NaN,4.0,16.0
189,TXN_2643815,Cookie,NaN,1.0,3.0
198,TXN_9788956,Juice,NaN,3.0,12.0


Jumlah Quantity kosong: 479
Quantity kosong dengan informasi pendukung lengkap: 400


## 15. Memulihkan Quantity Berdasarkan Total Transaksi

Nilai `Quantity` yang kosong akan dihitung dari `Total Spent` dibagi `Price Per Unit`, tetapi hanya jika kedua nilai tersedia, harga lebih besar dari nol, dan hasil pembagian merupakan bilangan bulat positif.

Jika kondisi tersebut tidak terpenuhi, nilai `Quantity` akan tetap kosong agar tidak menghasilkan jumlah barang yang tidak masuk akal.

In [20]:
# Memilih transaksi yang Quantity-nya kosong
# serta memiliki harga dan total transaksi
kondisi_quantity = (
    df_clean["Quantity"].isna()
    & df_clean["Price Per Unit"].notna()
    & df_clean["Total Spent"].notna()
    & (df_clean["Price Per Unit"] > 0)
)

# Menghitung perkiraan Quantity
hasil_bagi = (
    df_clean.loc[kondisi_quantity, "Total Spent"]
    / df_clean.loc[kondisi_quantity, "Price Per Unit"]
)

# Hanya mengambil hasil berupa bilangan bulat positif
quantity_valid = (
    (hasil_bagi > 0)
    & np.isclose(hasil_bagi, hasil_bagi.round())
)

indeks_valid = hasil_bagi[quantity_valid].index

# Mengisi Quantity yang dapat dihitung secara logis
df_clean.loc[indeks_valid, "Quantity"] = (
    hasil_bagi.loc[indeks_valid].round()
)

print("Quantity berhasil dipulihkan:", len(indeks_valid))
print("Quantity kosong setelah pemulihan:", df_clean["Quantity"].isna().sum())

Quantity berhasil dipulihkan: 456
Quantity kosong setelah pemulihan: 23


## 16. Memeriksa Nama Produk yang Kosong

Kolom `Item` menunjukkan jenis produk yang dibeli. Nilai kosong akan diperiksa berdasarkan informasi transaksi lainnya. Nama produk tidak akan ditentukan hanya berdasarkan harga karena beberapa produk memiliki harga yang sama.

Jika nama produk tidak dapat dipastikan dari data yang tersedia, nilai tersebut akan tetap kosong untuk menghindari kesalahan klasifikasi.

In [22]:
# Melihat transaksi dengan nama produk kosong
item_kosong = df_clean[df_clean["Item"].isna()]

display(
    item_kosong[
        [
            "Transaction ID",
            "Item",
            "Quantity",
            "Price Per Unit",
            "Total Spent",
            "Payment Method",
            "Location",
            "Transaction Date"
        ]
    ].head(15)
)

print("Jumlah Item kosong:", df_clean["Item"].isna().sum())

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
6,TXN_4433211,NaN,3.0,3.0,9.0,NaN,Takeaway,2023-10-06
8,TXN_4717867,NaN,5.0,3.0,15.0,NaN,Takeaway,2023-07-28
14,TXN_8915701,NaN,2.0,1.5,3.0,NaN,In-store,2023-03-21
30,TXN_1736287,NaN,5.0,2.0,10.0,Digital Wallet,NaN,2023-06-02
31,TXN_8927252,NaN,2.0,1.0,2.0,Credit Card,NaN,2023-11-06
33,TXN_7710508,NaN,5.0,1.0,5.0,Cash,NaN,NaT
36,TXN_6855453,NaN,4.0,3.0,12.0,NaN,In-store,2023-07-17
52,TXN_8914892,NaN,5.0,5.0,25.0,Digital Wallet,NaN,2023-03-15
61,TXN_8051289,NaN,1.0,3.0,3.0,NaN,In-store,2023-10-09
63,TXN_9099694,NaN,3.0,5.0,15.0,NaN,Takeaway,2023-11-18


Jumlah Item kosong: 969


## 17. Evaluasi Nilai Kosong Setelah Pemulihan

Setelah memulihkan nilai `Total Spent`, `Price Per Unit`, dan `Quantity` yang dapat dihitung, dilakukan pemeriksaan ulang untuk mengetahui jumlah nilai kosong yang masih tersisa pada setiap kolom. Hasil pemeriksaan digunakan untuk menentukan langkah penanganan selanjutnya.

In [23]:
# Menghitung nilai kosong setelah proses pemulihan
ringkasan_akhir = pd.DataFrame({
    "Jumlah Kosong": df_clean.isna().sum(),
    "Persentase Kosong (%)": (
        df_clean.isna().mean() * 100
    ).round(2)
})

display(ringkasan_akhir)

print("Ukuran dataset saat ini:", df_clean.shape)

,Jumlah Kosong,Persentase Kosong (%)
Transaction ID,0,0.00
Item,969,9.69
Quantity,23,0.23
Price Per Unit,54,0.54
Total Spent,40,0.40
Payment Method,3178,31.78
Location,3961,39.61
Transaction Date,460,4.60


Ukuran dataset saat ini: (10000, 8)


## 18. Memvalidasi Konsistensi Setelah Pemulihan

Setelah nilai kosong yang dapat dipulihkan ditangani, dilakukan pemeriksaan ulang untuk memastikan bahwa nilai `Total Spent` tetap sesuai dengan hasil perkalian `Quantity` dan `Price Per Unit`. Pemeriksaan hanya menggunakan transaksi yang memiliki ketiga nilai tersebut.

In [24]:
# Memilih transaksi dengan Quantity, harga, dan total lengkap
data_validasi = df_clean.dropna(
    subset=["Quantity", "Price Per Unit", "Total Spent"]
).copy()

# Menghitung total berdasarkan jumlah dan harga
data_validasi["Calculated Total"] = (
    data_validasi["Quantity"]
    * data_validasi["Price Per Unit"]
)

# Membandingkan total tercatat dengan hasil perhitungan
data_validasi["Total Match"] = np.isclose(
    data_validasi["Total Spent"],
    data_validasi["Calculated Total"]
)

print("Transaksi yang divalidasi:", len(data_validasi))
print("Total sesuai:", data_validasi["Total Match"].sum())
print("Total tidak sesuai:", (~data_validasi["Total Match"]).sum())

Transaksi yang divalidasi: 9909
Total sesuai: 9909
Total tidak sesuai: 0


## 19. Menyimpan Dataset Hasil Pembersihan

Dataset yang telah melalui proses pembersihan dan validasi disimpan dalam file terpisah. Dataset asli tetap dipertahankan agar dapat digunakan kembali apabila diperlukan.

File hasil pembersihan akan digunakan pada tahap analisis data berikutnya.

In [25]:
from pathlib import Path

# Membuat folder penyimpanan jika belum tersedia
folder_output = Path("../data/processed")
folder_output.mkdir(parents=True, exist_ok=True)

# Menyimpan dataset hasil pembersihan
file_output = folder_output / "cafe_sales_clean.csv"
df_clean.to_csv(file_output, index=False)

print("Dataset berhasil disimpan.")
print("Lokasi file:", file_output)
print("Ukuran dataset:", df_clean.shape)

Dataset berhasil disimpan.
Lokasi file: ..\data\processed\cafe_sales_clean.csv
Ukuran dataset: (10000, 8)


## 20. Kesimpulan Data Preparation

Pada tahap data preparation, dataset Dirty Cafe Sales yang terdiri dari 10.000 baris dan 8 kolom telah melalui proses pemeriksaan dan pembersihan. Nilai `UNKNOWN` dan `ERROR` diubah menjadi nilai kosong (`NaN`), kemudian tipe data numerik dan tanggal disesuaikan.

Pemulihan dilakukan pada nilai `Total Spent`, `Price Per Unit`, dan `Quantity` yang dapat dihitung berdasarkan informasi transaksi yang tersedia. Hasil validasi menunjukkan bahwa 9.909 transaksi dengan nilai lengkap pada ketiga kolom tersebut memiliki total transaksi yang konsisten.

Beberapa nilai kosong masih dipertahankan karena tidak dapat dipulihkan secara pasti. Dataset hasil pembersihan disimpan sebagai `data/processed/cafe_sales_clean.csv` untuk digunakan pada tahap analisis berikutnya.